# Block 10: Assemble the tiny GPT

**Paper anchor:** Figure 1 (decoder half), §3.4 (shared weights, sqrt(d_model) scale), §3.5
**Builds on:** blocks 07 to 09; prerequisite P8
**Time:** about 5 hours, in short sessions
**You will write:** `TinyGPT`
**Done when:** the checkpoint passes and your hand parameter count matches PyTorch's

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b10
import torch, torch.nn as nn

## Big picture (Pass 2 for the decoder half of Fig. 1)
A GPT keeps Fig. 1's decoder stack and deletes its encoder-decoder attention sub-layer, since there is no encoder
(outside knowledge: Radford et al. 2018). What remains is block 09's layer with a causal mask.

| step | shape |
|---|---|
| idx | (B, T) |
| tok_emb(idx) * sqrt(d_model) (§3.4) | (B, T, d) |
| + positions (§3.5), dropout (§5.4) | (B, T, d) |
| N blocks with causal mask | (B, T, d) |
| [final LayerNorm if norm_first] | (B, T, d) |
| lm_head (tied to tok_emb, §3.4) | (B, T, V) |

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. Which box of Fig. 1 do we delete, and why is that safe for a pure language model?
2. With tied weights, how many parameters do the embedding and output layers cost for V = 65, d = 128? Untied?
3. The paper multiplies embeddings by sqrt(d_model) but never says why. Propose a reason (label it as interpretation).

*My answers:*

1. 
2. 
3. 

In [ ]:
from minitransformer.numpy_ops import sinusoidal_pe        # graduated in block 08
from minitransformer.layers import TransformerBlock        # graduated in block 09

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

class TinyGPT(nn.Module):
    """Decoder-only Transformer language model.

    Fig. 1's decoder stack without the encoder-decoder attention sub-layer (that deletion is
    outside the paper: Radford et al. 2018). Each layer is a TransformerBlock with a causal mask.

    __init__(vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1,
             pos="sinusoidal", tie_weights=True, norm_first=False)
        Attributes:
          tok_emb:  nn.Embedding(vocab_size, d_model)
          lm_head:  nn.Linear(d_model, vocab_size, bias=False)
          max_len:  int, longest context the model accepts
        pos: "sinusoidal" (§3.5, a fixed buffer), "learned" (nn.Embedding(max_len, d_model), Table 3 row E),
             or "none" (no positional information; for the ablation in block 13)
        tie_weights=True: lm_head shares tok_emb's weight matrix (§3.4)
        norm_first is passed to every TransformerBlock; norm_first=True also adds one final
             LayerNorm before lm_head (pre-norm convention, outside the paper)

    forward(idx) -> logits
        idx: (B, T) int64 token ids, T <= max_len (raise ValueError otherwise, for every pos option)
        Steps: tok_emb(idx) * sqrt(d_model) (§3.4), add positions, dropout (§5.4),
               blocks with a causal mask, [final LayerNorm if norm_first], lm_head
        returns (B, T, vocab_size)
    """

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1,
                 pos="sinusoidal", tie_weights=True, norm_first=False):
        super().__init__()
        raise NotImplementedError("Block 10: your turn")

    def forward(self, idx):
        raise NotImplementedError("Block 10: your turn")

In [ ]:
b10.check_gpt(TinyGPT)

## Sanity experiments
1. Derive the parameter count for V = 65, d_model = 128, 4 heads, 4 layers, d_ff = 512 by hand. Compare with `sum(p.numel() ...)`.
2. Before training, the loss should be near ln V if the model starts out uncommitted. Predict, then measure on a random batch.
   If it is far above ln V, we will discuss initialization in chat (outside the paper: GPT-2 uses std 0.02).

In [ ]:
# YOUR CODE

## Explain it back
Without notes. Then paste your answers into chat.

1. Map every line of your forward() to a section or equation of the paper.
2. What does weight tying assume about the input and output vocabularies?
3. Why is the causal mask the only thing that makes this a language model rather than an encoder?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
Copy the class into `src/minitransformer/model.py`; `pytest -q`; commit `block 10: tiny GPT`. Mark P8 done.